In [1]:
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain.tools import tool
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

C:\Users\v-kpurwar\AppData\Local\Temp\ipykernel_193408\1635441099.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun, ArxivQueryRun


True

### Define LLM

In [32]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

### Tool Creation

In [3]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    """Use this tool to search the web"""
    search = DuckDuckGoSearchRun()
    response =  search.invoke(query)
    return response


@tool
def tool_wikipedia_search(query: str) -> str:
    """Use this tool to get the results from Wikipedia"""
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    response = wikipedia.run(query)
    return response

@tool
def tool_arxiv_search(query: str) -> str:
    """Use this tool to get the research papers"""
    api_wrapper = ArxivAPIWrapper(
        doc_content_chars_max=2000,
        top_k_results=3
    )
    arxiv = ArxivQueryRun(api_wrapper=api_wrapper)
    response = arxiv.run(query)
    return response


#### Create a custom tool

In [10]:
@tool
def tool_personl_info(query: str) -> str:
    """Use this tool when you need to answer questions about personal information"""
    infos = [
        {
            "name" : "John Doe",
            "age" : 30,
            "role": "Software Engineer"
        },
        {
            "name" : "Jane Smith",
            "age" : 28,
            "role": "Data Scientist"
        },
        {
            "name" : "Alice Johnson",
            "age" : 35,
            "role": "Product Manager"
        }
    ]
    
    for info in infos:
        if info['name'].lower() in query.lower():
            return f"Name: {info['name']}, Age: {info['age']}, Role: {info['role']}"
    return "No matching personal information found."


#### Create a RAG tool

In [15]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSphere Org document"""

    # embedding model
    embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

    #  connection
    chroma_db_conn = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

    ## retrieve relevant documents from the vector store
    retriever = chroma_db_conn.similarity_search(query, k=3)
    relevant_docs = "\n".join([doc.page_content for doc in retriever])
    return relevant_docs

### Create Toolkit

In [16]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personl_info,
    tool_rag
]

### Tool Binding

In [35]:
llm_bind = llm.bind_tools(toolkit)

In [36]:
llm_bind.invoke("When was Novasphere Org founded and what is its mission?") ## LLM suggests to use the tool and gets the answer from it.

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 16.403528461s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

### Loop and Tool Execution using Agent

In [19]:
# An agent is a model calling tools in a loop until a given task is complete.

from langchain.agents import create_agent

my_agent = create_agent(llm_bind, toolkit)

In [20]:
my_agent.invoke(
    {"messages": [
        {"role": "user", "content": "When was Novasphere Org founded and what is its mission?"}
        ]
    }
) 

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 33.428912457s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT